# UCLA Admission Prediction
Predicting graduate admission probability from academic scores.

## Project Overview
Regression project to predict the chance of admission to UCLA based on GRE, TOEFL, CGPA, and other academic factors.

## Learning Objectives
- Work with small, clean tabular data
- Build regression models for probability prediction
- Compare linear vs boosting models on small datasets

## Problem Statement
Given a student's GRE score, TOEFL score, university rating, SOP, LOR, CGPA, and research experience, predict their chance of admission (0-1 probability).

## Why This Project Matters
Helps students assess their admission chances and universities identify key predictive factors in admissions.

## Dataset Overview
| Feature | Value |
|---|---|
| **Source** | Local: admission_predict.csv |
| **Target** | Chance of Admit (0-1) |
| **Rows** | ~400 |
| **Features** | GRE, TOEFL, University Rating, SOP, LOR, CGPA, Research |

## Dataset Source & License
UCLA admission dataset from Kaggle (mohan0040/mit-admission). Public domain educational data.

## Environment Setup

In [ ]:
import subprocess, sys
for pkg in ['catboost','lightgbm','xgboost','flaml','lazypredict']:
    try: __import__(pkg)
    except ImportError: subprocess.check_call([sys.executable,'-m','pip','install',pkg,'-q'])

## Imports

In [ ]:
import os, json, warnings, numpy as np, pandas as pd, matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import r2_score, mean_absolute_error, root_mean_squared_error
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
warnings.filterwarnings('ignore')
SEED = 42
np.random.seed(SEED)
TEST_SIZE = 0.2
SAVE_DIR = os.path.dirname(os.path.abspath('__file__'))
os.makedirs(SAVE_DIR, exist_ok=True)

## Configuration

In [ ]:
MAX_ROWS = 50000

## Dataset Loading

In [ ]:
csv_path = os.path.join(SAVE_DIR, 'admission_predict.csv')
if not os.path.exists(csv_path):
    raise FileNotFoundError(f'Dataset not found at {csv_path}')
df = pd.read_csv(csv_path)
# Clean column names
df.columns = df.columns.str.strip()
print(f'Shape: {df.shape}')
print(df.columns.tolist())
df.head()

## Data Validation

In [ ]:
print('Missing values:')
print(df.isnull().sum())
print(f'\nDuplicates: {df.duplicated().sum()}')

# Find target
target_candidates = [c for c in df.columns if 'admit' in c.lower() or 'chance' in c.lower()]
TARGET = target_candidates[0] if target_candidates else df.select_dtypes('number').columns[-1]
print(f'\nTarget: {TARGET}')
print(df[TARGET].describe())

## Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
df[TARGET].hist(bins=30, ax=axes[0,0], edgecolor='black')
axes[0,0].set_title('Admission Chance Distribution')

for i, col in enumerate(['GRE Score', 'TOEFL Score', 'CGPA']):
    if col in df.columns:
        ax = axes[(i+1)//2, (i+1)%2]
        ax.scatter(df[col], df[TARGET], alpha=0.5, s=10)
        ax.set_xlabel(col)
        ax.set_ylabel(TARGET)
        ax.set_title(f'{col} vs Admission Chance')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'eda_plots.png'), dpi=100)
plt.show()

## Target Analysis

In [ ]:
print(df[TARGET].describe())
print(f'\nSkewness: {df[TARGET].skew():.2f}')

## Train/Test Split

In [ ]:
# Drop serial number if present
drop_cols = [c for c in df.columns if 'serial' in c.lower() or c.lower() == 'no']
df = df.drop(columns=drop_cols, errors='ignore')

X = df.drop(columns=[TARGET])
y = df[TARGET]

# All features are numeric for this dataset
X = X.fillna(X.median())

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=TEST_SIZE, random_state=SEED)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')

## Preprocessing
All features are numeric. Dropped serial number. Filled any missing with median.

## Feature Engineering
Using raw academic scores. Small dataset — minimal feature engineering to avoid overfitting.

## Baseline Model

In [ ]:
bl = LinearRegression()
bl.fit(X_train, y_train)
bl_pred = bl.predict(X_test)
print(f'Baseline LR: R2={r2_score(y_test, bl_pred):.4f}  RMSE={root_mean_squared_error(y_test, bl_pred):.4f}  MAE={mean_absolute_error(y_test, bl_pred):.4f}')

## LazyPredict Benchmark

In [ ]:
# --- LazyPredict Benchmark ---
try:
    from lazypredict.Supervised import LazyRegressor
    n_lp = min(5000, len(X_train))
    lr = LazyRegressor(verbose=0, ignore_warnings=True)
    lp_models, _ = lr.fit(X_train.head(n_lp), X_test.head(min(1000, len(X_test))),
                          y_train.head(n_lp), y_test.head(min(1000, len(y_test))))
    print(lp_models.head(15))
except Exception as e:
    print(f"LazyPredict skipped: {e}")

## FLAML AutoML

In [ ]:
# --- FLAML AutoML ---
try:
    from flaml import AutoML
    automl = AutoML()
    automl.fit(X_train, y_train, task='regression', time_budget=60, seed=SEED, verbose=0)
    flaml_pred = automl.predict(X_test)
    from sklearn.metrics import r2_score, root_mean_squared_error, mean_absolute_error
    print(f"FLAML best: {automl.best_estimator}")
    print(f"  R2={r2_score(y_test, flaml_pred):.4f}  RMSE={root_mean_squared_error(y_test, flaml_pred):.4f}  MAE={mean_absolute_error(y_test, flaml_pred):.4f}")
except Exception as e:
    print(f"FLAML skipped: {e}")

## Additional Models: CatBoost, LightGBM, XGBoost

In [ ]:
# --- Boosting Models ---
models = {}
cb = CatBoostRegressor(iterations=500, learning_rate=0.05, depth=6, random_seed=SEED, verbose=0)
cb.fit(X_train, y_train)
models['CatBoost'] = cb

lgb = LGBMRegressor(n_estimators=500, learning_rate=0.05, max_depth=6, random_state=SEED, verbose=-1)
lgb.fit(X_train, y_train)
models['LightGBM'] = lgb

xgb_m = XGBRegressor(n_estimators=500, learning_rate=0.05, max_depth=6, random_state=SEED, verbosity=0)
xgb_m.fit(X_train, y_train)
models['XGBoost'] = xgb_m

for name, m in models.items():
    p = m.predict(X_test)
    print(f"{name}: R2={r2_score(y_test, p):.4f}  RMSE={root_mean_squared_error(y_test, p):.4f}  MAE={mean_absolute_error(y_test, p):.4f}")

## Top 3 Model Selection

In [ ]:
# --- Top 3 Selection ---
all_results = {}
for name, m in models.items():
    p = m.predict(X_test)
    all_results[name] = {'R2': r2_score(y_test, p), 'RMSE': root_mean_squared_error(y_test, p), 'MAE': mean_absolute_error(y_test, p)}

results_df = pd.DataFrame(all_results).T.sort_values('R2', ascending=False)
print(results_df)
top3_names = results_df.head(3).index.tolist()
print(f'\nTop 3: {top3_names}')

## Final Evaluation of Top 3

In [ ]:
# --- Final Evaluation of Top 3 ---
final_results = {}
for name in top3_names:
    m = models[name]
    p = m.predict(X_test)
    r2 = r2_score(y_test, p)
    rmse = root_mean_squared_error(y_test, p)
    mae = mean_absolute_error(y_test, p)
    final_results[name] = {'R2': r2, 'RMSE': rmse, 'MAE': mae}
    print(f"{name}: R2={r2:.4f}  RMSE={rmse:.4f}  MAE={mae:.4f}")

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
names = list(final_results.keys())
for i, metric in enumerate(['R2', 'RMSE', 'MAE']):
    vals = [final_results[n][metric] for n in names]
    axes[i].bar(names, vals, color=['#2ecc71','#3498db','#e74c3c'])
    axes[i].set_title(metric)
    axes[i].tick_params(axis='x', rotation=15)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'top3_comparison.png'), dpi=100)
plt.show()

## Error Analysis

In [ ]:
# --- Error Analysis ---
best_name = top3_names[0]
best_model = models[best_name]
preds = best_model.predict(X_test)
residuals = y_test - preds

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].hist(residuals, bins=50, edgecolor='black')
axes[0].set_title('Residual Distribution')
axes[0].axvline(0, color='red', linestyle='--')
axes[1].scatter(preds, residuals, alpha=0.3, s=5)
axes[1].axhline(0, color='red', linestyle='--')
axes[1].set_title('Residuals vs Predicted')
axes[2].scatter(y_test, preds, alpha=0.3, s=5)
mn, mx = y_test.min(), y_test.max()
axes[2].plot([mn, mx], [mn, mx], 'r--')
axes[2].set_title('Actual vs Predicted')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'error_analysis.png'), dpi=100)
plt.show()

## Interpretation & Business Insight
CGPA and GRE score are the strongest predictors of admission. Research experience provides a moderate boost. University rating matters more than SOP/LOR scores.

## Limitations
- Very small dataset (400 rows)
- May not generalize to other universities
- No temporal component (year of application)

## How to Improve
- Collect data from multiple universities
- Add more features (extracurriculars, work experience)
- Use ordinal encoding for university rating

## Production Considerations
- Would need constant retraining as admission criteria change
- Must be transparent about prediction uncertainty
- Should not be sole decision factor

## Common Mistakes
- Treating this as a classification problem (target is continuous)
- Using serial number as a feature
- Ignoring multicollinearity between GRE/TOEFL/CGPA

## Mini Challenge
1. Binarize the target at 0.5 and try classification
2. Try polynomial features for GRE × CGPA
3. Compare Ridge vs Lasso regularization

## Final Summary
Predicted admission probability from academic scores. Linear models perform well on this small, well-behaved dataset.

In [ ]:
# --- Save Metrics ---
metrics = {name: {k: round(v, 4) for k, v in vals.items()} for name, vals in final_results.items()}
metrics['best_model'] = top3_names[0]
with open(os.path.join(SAVE_DIR, 'metrics.json'), 'w') as f:
    json.dump(metrics, f, indent=2)
print('Saved metrics.json')
print(json.dumps(metrics, indent=2))